# 02 ViViT 年度滚动训练与因子生成

本 Notebook 是统一的训练参数入口和进度展示平台。模型、数据、中性化目标、联合 Barra R² 正则、对抗式不变表征、Barra 条件门控、严格输出中性化、Alpha/Style 双塔、训练、后台任务管理、验证与报告逻辑均位于 Python 后端；Notebook 不定义函数或类。正式训练默认运行在独立的 detached screen 中，Notebook kernel 退出不会连带终止训练。


## 步骤 1：选择实验模式与训练超参数

当前统一入口覆盖七类已实现实验：No.0 Baseline、No.1 Neutral（Y/X/XY）、No.2 raw-score 联合 Barra R² 惩罚、No.3 对抗式 Barra 不变表征、No.4 Barra 条件化乘法门控、No.5 条件化严格输出中性化和 No.6 Barra 条件化 Alpha/Style 双塔。

- **No.0 Baseline**：使用原始标签和原始分数，不读取 Barra。
- **No.1 Neutral**：可对标签、训练分数或二者做十个连续 Barra 变量的残差化。
- **No.2 Barra R²**：使用原始标签，并惩罚 raw score 对十变量与行业固定效应的联合回归 R²。
- **No.3 对抗式 Barra**：Barra 只监督训练期 CLS 对抗头，不进入收益模型输入。
- **No.4 条件门控**：Alpha 模型读取 Barra 条件，使用完整 Barra 残差标签，输出不投影。
- **No.5 严格输出中性化**：在 No.4 条件分数后加入完整 Barra 可微投影，发布严格中性信号。
- **No.6 Alpha/Style 双塔**：Alpha 塔读取视频和可选 Barra 条件，Style 塔只读取 Barra；总分数学习原始收益，Alpha 与 Style 分别学习完整残差和拟合收益，正式发布未经投影的 Alpha 分数。

No.6 的总损失是四项之和：Total 对原始标签的 Soft RankIC、Alpha 对完整残差标签的 Soft RankIC、Style 对拟合收益标签的 Soft RankIC，以及 Alpha 对真实 Barra/行业的联合 R² 惩罚。早停只看验证集 Alpha 对完整残差标签的日均 Spearman RankIC。`real/alpha_video_only/shuffled/zero` 只改变两塔输入；标签分解、R² 和暴露诊断始终使用真实 Barra。

参数单元已按使用频率排序：实验模式与 run 名、启动控制、GPU 资源、核心优化参数在前，特定结构和性能参数在后。每个可调项的注释均列出当前全部合法选项或数值范围。


In [1]:
from pathlib import Path
import subprocess
import sys

VIVIT_DIR = Path.cwd().resolve()
if VIVIT_DIR.name != "ViViT":
    VIVIT_DIR = VIVIT_DIR / "ViViT"
if str(VIVIT_DIR) not in sys.path:
    sys.path.insert(0, str(VIVIT_DIR))
from stage07_training_jobs import (
    default_screen_session_name, format_training_job_status,
    launch_training_job, tail_training_job_log, training_job_status,
)
from stage04_experiments import resolve_experiment
PYTHON = Path("/root/miniconda3/bin/python")
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

# ===== 1. 重要：实验模式与 run 名 =====
# 可选："no0"、"no1_y"、"no1_x"、"no1_xy"、"no2"、"no3"、"no4"、"no5"、"no6"。
# 方案编号决定结构开关；下方仅保留当前方案真正使用的强度和结构超参数。
EXPERIMENT_ID = "no6"
# 可选：False、True；No.2/No.3/No.4/No.5/No.6 要求二者同时为 False。
RESIDUALIZE_TARGET = False
PROJECT_MODEL_OUTPUT = False
# 可选：0.0=关闭；建议路径为 0.0、0.10、0.30，且允许任意 >=0 浮点数。
# 大于 0 时惩罚 raw_score 对十个连续风格/反转变量+行业固定效应的联合 R²。
RAW_BARRA_R2_LAMBDA = 0.30
# 可选："off"、"video_only"、"real"、"shuffled"、"zero"。
# 非 off 时开启 No.4；默认 real 使用与股票正确配对的 T 日 Barra。
BARRA_CONDITIONING_MODE = "off"
# 可选：False、True；True=开启 No.5，在条件分数后严格投影掉十风格和行业固定效应。
# No.5 使用等权 W=I，要求 RAW_BARRA_R2_LAMBDA=0；正式发布投影后信号。
BARRA_OUTPUT_NEUTRALIZATION = False
# 可选：False、True；True=开启 No.6 Alpha/Style 双塔。
BARRA_DUAL_TOWER = True
# 可选："real"、"alpha_video_only"、"shuffled"、"zero"。
# real=两塔读真实 Barra；alpha_video_only=Alpha 绕过门控、Style 读真实 Barra；
# shuffled=两塔读同一份逐日确定性联合置乱 Barra；zero=两塔条件输入全部置零。
BARRA_DUAL_TOWER_MODE = "real"
# 可选：任意 >0 浮点数；Alpha 残差 RankIC 损失权重 eta。
DUAL_TOWER_ALPHA_LOSS_WEIGHT = 1.0
# 可选：任意 >0 浮点数；Style 拟合收益 RankIC 损失权重 mu。
DUAL_TOWER_STYLE_LOSS_WEIGHT = 1.0
# 可选：任意正整数；独立 Style MLP 两层隐藏维度。
DUAL_TOWER_STYLE_HIDDEN_DIM = 64
# 可选：[0,1) 浮点数；独立 Style MLP 的 Dropout。
DUAL_TOWER_STYLE_DROPOUT = 0.10
# 可选：任意正整数；Barra MLP 隐藏层及 z_B 的维度。
BARRA_GATE_HIDDEN_DIM = 64
# 可选：任意正整数；UNKNOWN=0 的 embedding 始终固定为零。
BARRA_INDUSTRY_EMBEDDING_DIM = 16
# 可选：[0,1) 浮点数；Barra MLP 隐藏层 Dropout。
BARRA_GATE_DROPOUT = 0.10
# 可选：(0,1) 浮点数；通道缩放范围为 (1-scale, 1+scale)。
BARRA_GATE_SCALE = 0.50
# 可选：False、True；True=开启 No.3，对抗监督不作为收益模型输入。
ADVERSARIAL_BARRA = False
# 可选：任意 >0 浮点数；GRL 作用于编码器的最大反向梯度系数。
ADVERSARIAL_GAMMA_MAX = 0.10
# 可选：任意 >=0 整数；前若干 epoch 的 gamma=0，仅训练对抗头。
ADVERSARIAL_WARMUP_EPOCHS = 2
# 可选：任意 >=1 整数；warmup 后余弦升至 gamma_max 的 epoch 数。
ADVERSARIAL_RAMP_EPOCHS = 5
# 可选：任意 >0 浮点数；十个连续风格 MaskedMSE 的损失权重。
ADVERSARIAL_STYLE_LOSS_WEIGHT = 1.0
# 可选：任意 >0 浮点数；32 行业加权 CE/log(32) 的损失权重。
ADVERSARIAL_INDUSTRY_LOSS_WEIGHT = 1.0
# 可选：任意正整数；两个独立对抗头的隐藏维度。
ADVERSARIAL_HEAD_HIDDEN_DIM = 128
# 可选：[0,1) 浮点数；两个对抗头隐藏层的 Dropout。
ADVERSARIAL_HEAD_DROPOUT = 0.10
# 可选：None=按模式与 λ 自动命名；任意非空字符串=显式指定新 run 名。
RUN_NAME_OVERRIDE = "dual_tower_ddp"

# ===== 2. 重要：启动、恢复与日志 =====
# 可选："a2"=每卡并发独立任务；"ddp"=单个年度×seed 使用 DEVICE_IDS 全部 GPU。
TRAINING_ROUTE = "ddp"
# 可选：False、True；True=启动或恢复正式训练，False=只准备配置和查看已有状态。
RUN_FULL_TRAINING = True
# 可选："background"=推荐，训练脱离 Notebook kernel；"foreground"=仅调试，kernel 中断会终止训练。
TRAIN_LAUNCH_MODE = "background"
# 可选：None=自动使用 vivit_<RUN_NAME>；任意合法非空字符串=显式指定独立 screen 名。
# 合法字符仅包括字母、数字、下划线、连字符和句点。
SCREEN_SESSION_NAME = None
# 可选：0=复用同签名完成任务/从 last.pt 续训；1=显式忽略旧状态重新训练。普通恢复必须为 0。
OVERWRITE = 0
# 可选：False、True；是否先运行 2018/seed42 的单 batch 前向、反向和验证 smoke。
RUN_SMOKE_TEST = True
# 可选：任意 >=0 整数；状态单元显示后台总控日志末尾行数，0=不显示日志。
STATUS_LOG_TAIL_LINES = 0

# ===== 3. 重要：GPU 与数据加载资源 =====
# 可选："0"、"0,1"、"0,1,2,3" 等逗号分隔且不重复的物理 GPU 编号。
DEVICE_IDS = "0,1,2,3"
# 可选：任意正整数；每张 GPU 的并发年度×seed 任务数，显存不足时使用 1。
TASKS_PER_GPU = 1
# 可选：任意 >=0 整数；0 表示在训练主进程读取数据。
WORKERS_PER_TASK = 8

# ===== 4. 重要：训练目标与优化 =====
# 可选："fixed_samples"、"trade_date"；No.1–No.6 只允许 "trade_date"。
TRAIN_BATCH_MODE = "trade_date"
# 可选："pearson_ic"、"sigmoid_rank_ic"、"neural_sort_rank_ic"、"mse"、"huber"、"smooth_l1"、"pearson_huber"、"sigmoid_rank_ic_turnover"；No.2–No.6 固定 "sigmoid_rank_ic"。
LOSS_NAME = "sigmoid_rank_ic"
# 可选："daily_mean_pearson_ic"、"daily_mean_spearman_ic"；No.2–No.6 固定后者。
VALIDATION_METRIC = "daily_mean_spearman_ic"
# 可选：任意 >0 浮点数；Adam 学习率。
LEARNING_RATE = 0.001
# 可选：任意 >=1 整数；最大训练 epoch。
MAX_EPOCHS = 100
# 可选：任意 >=1 整数；验证 RankIC 连续不改善时的早停容忍 epoch 数。
PATIENCE = 10
# 可选：任意 >0 浮点数；用于 sigmoid/neural-sort/turnover RankIC 损失。
RANK_TEMPERATURE = 0.1

# ===== 5. 次要：特定损失参数与运行性能 =====
# 可选：任意 >=0 浮点数；仅 sigmoid_rank_ic_turnover 使用。
TURNOVER_LAMBDA = 0.1
# 可选：任意 >0 浮点数；用于 huber 和 pearson_huber。
HUBER_DELTA = 1.0
# 可选：任意 >0 浮点数；仅 smooth_l1 使用。
SMOOTH_L1_BETA = 0.1
# 可选：[0,1] 浮点数；pearson_huber 中 Pearson 项权重。
HYBRID_WEIGHT = 0.5
# 可选：任意 >1 整数；仅 Baseline fixed_samples 训练使用。
FIXED_BATCH_SIZE = 5000
# 可选：任意 >1 整数；仅 Baseline validation/prediction 分批读取使用。
INFERENCE_BATCH_SIZE = 5000
# 可选：0=关闭、1=开启 torch.compile。
USE_TORCH_COMPILE = 1
# 可选："default"、"lite"、"reduce-overhead"、"max-autotune-no-cudagraphs"、"max-autotune"。
COMPILE_MODE = "reduce-overhead"
# 可选：0=普通 Adam、1=CUDA fused Adam（CPU 自动回退）。
USE_FUSED_ADAM = 1
# 可选：0=关闭、1=独立 CUDA stream 预取。
USE_CUDA_PREFETCH = 1

# 显式方案编号统一安装结构开关，避免多个布尔量拼出未定义实验。
EXPERIMENT_SPEC = resolve_experiment(EXPERIMENT_ID)
EXPERIMENT_SWITCHES = {
    "no0": (False, False, False, "off", False, False),
    "no1_y": (True, False, False, "off", False, False),
    "no1_x": (False, True, False, "off", False, False),
    "no1_xy": (True, True, False, "off", False, False),
    "no2": (False, False, False, "off", False, False),
    "no3": (False, False, True, "off", False, False),
    "no4": (False, False, False, BARRA_CONDITIONING_MODE, False, False),
    "no5": (False, False, False, BARRA_CONDITIONING_MODE, True, False),
    "no6": (False, False, False, "off", False, True),
}
(RESIDUALIZE_TARGET, PROJECT_MODEL_OUTPUT, ADVERSARIAL_BARRA,
 BARRA_CONDITIONING_MODE, BARRA_OUTPUT_NEUTRALIZATION,
 BARRA_DUAL_TOWER) = EXPERIMENT_SWITCHES[EXPERIMENT_ID]
if EXPERIMENT_ID in {"no0", "no1_y", "no1_x", "no1_xy", "no3", "no5"} and RAW_BARRA_R2_LAMBDA != 0:
    raise ValueError(f"{EXPERIMENT_ID} 要求 RAW_BARRA_R2_LAMBDA=0")
if EXPERIMENT_ID == "no2" and RAW_BARRA_R2_LAMBDA <= 0:
    raise ValueError("no2 要求 RAW_BARRA_R2_LAMBDA>0")
if EXPERIMENT_ID in {"no4", "no5"} and BARRA_CONDITIONING_MODE == "off":
    raise ValueError(f"{EXPERIMENT_ID} 要求选择非 off 的 BARRA_CONDITIONING_MODE")
print({
    "experiment_id": EXPERIMENT_SPEC.experiment_id,
    "title": EXPERIMENT_SPEC.title,
    "barra_as_model_input": EXPERIMENT_SPEC.uses_barra_as_model_input,
    "barra_in_loss": EXPERIMENT_SPEC.uses_barra_in_loss,
})
MODE_RUN_NAMES = {
    (False, False): "sigmoid_daybatch_rankic",
    (True, False): "sigmoid_daybatch_rankic_style_residual_label",
    (False, True): "sigmoid_daybatch_rankic_style_projected_output",
    (True, True): "sigmoid_daybatch_rankic_style_residual_label_projected_output",
}
MODE = (bool(RESIDUALIZE_TARGET), bool(PROJECT_MODEL_OUTPUT))
if RAW_BARRA_R2_LAMBDA < 0:
    raise ValueError("RAW_BARRA_R2_LAMBDA 必须 >= 0")
BARRA_CONDITIONING_MODES = {"off", "video_only", "real", "shuffled", "zero"}
if BARRA_CONDITIONING_MODE not in BARRA_CONDITIONING_MODES:
    raise ValueError(f"BARRA_CONDITIONING_MODE 必须属于 {sorted(BARRA_CONDITIONING_MODES)}")
NO6_MODE = bool(BARRA_DUAL_TOWER)
NO4_MODE = BARRA_CONDITIONING_MODE != "off"
NO5_STRICT_MODE = bool(BARRA_OUTPUT_NEUTRALIZATION)
R2_PENALTY_ENABLED = RAW_BARRA_R2_LAMBDA > 0
R2_MODE = R2_PENALTY_ENABLED and not NO4_MODE and not NO6_MODE
BARRA_DUAL_TOWER_MODES = {"real", "alpha_video_only", "shuffled", "zero"}
if BARRA_DUAL_TOWER_MODE not in BARRA_DUAL_TOWER_MODES:
    raise ValueError(f"BARRA_DUAL_TOWER_MODE 必须属于 {sorted(BARRA_DUAL_TOWER_MODES)}")
ADVERSARIAL_MODE = bool(ADVERSARIAL_BARRA)
if R2_MODE and any(MODE):
    raise ValueError("No.2 联合 R² 不能与标签残差化或分数投影同时开启")
if R2_MODE and LOSS_NAME != "sigmoid_rank_ic":
    raise ValueError("No.2 联合 R² 当前只支持 sigmoid_rank_ic")
if R2_MODE and VALIDATION_METRIC != "daily_mean_spearman_ic":
    raise ValueError("No.2 联合 R² 早停只允许 daily_mean_spearman_ic")
if ADVERSARIAL_MODE and (any(MODE) or R2_PENALTY_ENABLED or NO4_MODE):
    raise ValueError("No.3 纯对抗实验不能与 Neutral、No.2 或 No.4 同时开启")
if ADVERSARIAL_MODE and (TRAIN_BATCH_MODE != "trade_date" or LOSS_NAME != "sigmoid_rank_ic"):
    raise ValueError("No.3 只允许 trade_date + sigmoid_rank_ic")
if ADVERSARIAL_MODE and VALIDATION_METRIC != "daily_mean_spearman_ic":
    raise ValueError("No.3 早停只允许 daily_mean_spearman_ic")
if ADVERSARIAL_MODE and (ADVERSARIAL_GAMMA_MAX <= 0 or ADVERSARIAL_WARMUP_EPOCHS < 0 or ADVERSARIAL_RAMP_EPOCHS < 1):
    raise ValueError("No.3 gamma 必须 >0、warmup 必须 >=0、ramp 必须 >=1")
if ADVERSARIAL_MODE and (ADVERSARIAL_STYLE_LOSS_WEIGHT <= 0 or ADVERSARIAL_INDUSTRY_LOSS_WEIGHT <= 0):
    raise ValueError("No.3 两个对抗损失权重必须 >0")
if ADVERSARIAL_MODE and (ADVERSARIAL_HEAD_HIDDEN_DIM < 1 or not 0 <= ADVERSARIAL_HEAD_DROPOUT < 1):
    raise ValueError("No.3 隐藏维度必须为正整数，Dropout 必须位于 [0,1)")
if NO4_MODE and (any(MODE) or ADVERSARIAL_MODE):
    raise ValueError("No.4 要求关闭 Neutral 投影和 No.3 对抗训练")
if NO4_MODE and (TRAIN_BATCH_MODE != "trade_date" or LOSS_NAME != "sigmoid_rank_ic"):
    raise ValueError("No.4 只允许 trade_date + sigmoid_rank_ic")
if NO4_MODE and VALIDATION_METRIC != "daily_mean_spearman_ic":
    raise ValueError("No.4 早停只允许 daily_mean_spearman_ic")
if NO4_MODE and (BARRA_GATE_HIDDEN_DIM < 1 or BARRA_INDUSTRY_EMBEDDING_DIM < 1):
    raise ValueError("No.4 隐藏维度和行业 embedding 维度必须为正整数")
if NO4_MODE and (not 0 <= BARRA_GATE_DROPOUT < 1 or not 0 < BARRA_GATE_SCALE < 1):
    raise ValueError("No.4 Dropout 必须位于 [0,1)，Gate scale 必须位于 (0,1)")
if NO5_STRICT_MODE and not NO4_MODE:
    raise ValueError("No.5 严格输出要求开启一种 Barra 条件门控模式")
if NO5_STRICT_MODE and R2_PENALTY_ENABLED:
    raise ValueError("No.5 已严格代数正交，RAW_BARRA_R2_LAMBDA 必须为0")
if NO6_MODE and (any(MODE) or ADVERSARIAL_MODE or NO4_MODE or NO5_STRICT_MODE):
    raise ValueError("No.6 要求关闭 Neutral、No.3、No.4 独立模式和 No.5 严格投影")
if NO6_MODE and (TRAIN_BATCH_MODE != "trade_date" or LOSS_NAME != "sigmoid_rank_ic"):
    raise ValueError("No.6 只允许 trade_date + sigmoid_rank_ic")
if NO6_MODE and VALIDATION_METRIC != "daily_mean_spearman_ic":
    raise ValueError("No.6 早停只允许 daily_mean_spearman_ic")
if NO6_MODE and (DUAL_TOWER_ALPHA_LOSS_WEIGHT <= 0 or DUAL_TOWER_STYLE_LOSS_WEIGHT <= 0):
    raise ValueError("No.6 的 eta 和 mu 必须 >0")
if NO6_MODE and (DUAL_TOWER_STYLE_HIDDEN_DIM < 1 or not 0 <= DUAL_TOWER_STYLE_DROPOUT < 1):
    raise ValueError("No.6 Style 隐藏维度必须为正整数，Dropout 必须位于 [0,1)")
STYLE_MODE = any(MODE) or R2_MODE or ADVERSARIAL_MODE or NO4_MODE or NO6_MODE
LAMBDA_SLUG = f"{RAW_BARRA_R2_LAMBDA:.2f}".replace(".", "p")
ETA_SLUG = f"{DUAL_TOWER_ALPHA_LOSS_WEIGHT:.2f}".replace(".", "p")
MU_SLUG = f"{DUAL_TOWER_STYLE_LOSS_WEIGHT:.2f}".replace(".", "p")
DEFAULT_RUN_NAME = (f"barra_dual_tower_{BARRA_DUAL_TOWER_MODE}_eta_{ETA_SLUG}_mu_{MU_SLUG}_r2_{LAMBDA_SLUG}" if NO6_MODE else (f"barra_gate_{BARRA_CONDITIONING_MODE}_strict_neutral" if NO5_STRICT_MODE else (f"barra_gate_{BARRA_CONDITIONING_MODE}_r2_{LAMBDA_SLUG}" if NO4_MODE else ("adversarial_barra_grl_0p10" if ADVERSARIAL_MODE else (f"sigmoid_daybatch_rankic_raw_barra_r2_lambda_{LAMBDA_SLUG}" if R2_MODE else MODE_RUN_NAMES[MODE])))))
BASE_RUN_NAME = RUN_NAME_OVERRIDE or DEFAULT_RUN_NAME
if TRAINING_ROUTE not in {"a2", "ddp"}:
    raise ValueError("TRAINING_ROUTE 只能是 a2 或 ddp")
RUN_NAME = BASE_RUN_NAME if TRAINING_ROUTE == "a2" else f"{BASE_RUN_NAME}_ddp"
if not isinstance(RUN_NAME, str) or not RUN_NAME.strip():
    raise ValueError("RUN_NAME 必须是非空字符串")
if TRAIN_LAUNCH_MODE not in {"background", "foreground"}:
    raise ValueError("TRAIN_LAUNCH_MODE 只能是 background 或 foreground")
if not isinstance(STATUS_LOG_TAIL_LINES, int) or STATUS_LOG_TAIL_LINES < 0:
    raise ValueError("STATUS_LOG_TAIL_LINES 必须是 >=0 的整数")
SCREEN_SESSION = SCREEN_SESSION_NAME or default_screen_session_name(RUN_NAME)
if STYLE_MODE and TRAIN_BATCH_MODE != "trade_date":
    raise ValueError("风格训练或 Barra R² 正则必须使用 trade_date batch")
if STYLE_MODE and LOSS_NAME == "sigmoid_rank_ic_turnover":
    raise ValueError("风格训练后端暂不支持 sigmoid_rank_ic_turnover")

BACKEND = VIVIT_DIR / ("stage06_barra_training.py" if STYLE_MODE else "stage05_baseline_training.py")
TRAINING_ARGS = [
    "--run-name", RUN_NAME,
    "--train-batch-mode", TRAIN_BATCH_MODE,
    "--loss-name", LOSS_NAME,
    "--rank-temperature", str(RANK_TEMPERATURE),
    "--turnover-lambda", str(TURNOVER_LAMBDA),
    "--huber-delta", str(HUBER_DELTA),
    "--smooth-l1-beta", str(SMOOTH_L1_BETA),
    "--hybrid-weight", str(HYBRID_WEIGHT),
    "--validation-metric", VALIDATION_METRIC,
    "--workers-per-task", str(WORKERS_PER_TASK),
    "--learning-rate", str(LEARNING_RATE),
    "--max-epochs", str(MAX_EPOCHS),
    "--patience", str(PATIENCE),
    "--compile", str(USE_TORCH_COMPILE),
    "--compile-mode", COMPILE_MODE,
    "--fused-adam", str(USE_FUSED_ADAM),
    "--cuda-prefetch", str(USE_CUDA_PREFETCH),
    "--overwrite", str(OVERWRITE),
]
if STYLE_MODE:
    TRAINING_ARGS += [
        "--residualize-target", str(int(RESIDUALIZE_TARGET)),
        "--project-model-output", str(int(PROJECT_MODEL_OUTPUT)),
        "--raw-barra-r2-lambda", str(RAW_BARRA_R2_LAMBDA),
        "--barra-conditioning-mode", BARRA_CONDITIONING_MODE,
        "--barra-output-neutralization", str(int(BARRA_OUTPUT_NEUTRALIZATION)),
        "--barra-dual-tower", str(int(BARRA_DUAL_TOWER)),
        "--barra-dual-tower-mode", BARRA_DUAL_TOWER_MODE,
        "--dual-tower-alpha-loss-weight", str(DUAL_TOWER_ALPHA_LOSS_WEIGHT),
        "--dual-tower-style-loss-weight", str(DUAL_TOWER_STYLE_LOSS_WEIGHT),
        "--dual-tower-style-hidden-dim", str(DUAL_TOWER_STYLE_HIDDEN_DIM),
        "--dual-tower-style-dropout", str(DUAL_TOWER_STYLE_DROPOUT),
        "--barra-gate-hidden-dim", str(BARRA_GATE_HIDDEN_DIM),
        "--barra-industry-embedding-dim", str(BARRA_INDUSTRY_EMBEDDING_DIM),
        "--barra-gate-dropout", str(BARRA_GATE_DROPOUT),
        "--barra-gate-scale", str(BARRA_GATE_SCALE),
        "--adversarial-barra", str(int(ADVERSARIAL_BARRA)),
        "--adversarial-gamma-max", str(ADVERSARIAL_GAMMA_MAX),
        "--adversarial-warmup-epochs", str(ADVERSARIAL_WARMUP_EPOCHS),
        "--adversarial-ramp-epochs", str(ADVERSARIAL_RAMP_EPOCHS),
        "--adversarial-style-loss-weight", str(ADVERSARIAL_STYLE_LOSS_WEIGHT),
        "--adversarial-industry-loss-weight", str(ADVERSARIAL_INDUSTRY_LOSS_WEIGHT),
        "--adversarial-head-hidden-dim", str(ADVERSARIAL_HEAD_HIDDEN_DIM),
        "--adversarial-head-dropout", str(ADVERSARIAL_HEAD_DROPOUT),
    ]
else:
    TRAINING_ARGS += [
        "--batch-size", str(FIXED_BATCH_SIZE),
        "--inference-batch-size", str(INFERENCE_BATCH_SIZE),
    ]

print({
    "training_route": TRAINING_ROUTE,
    "run_name": RUN_NAME,
    "backend": BACKEND.name,
    "residualize_target": RESIDUALIZE_TARGET,
    "project_model_output": PROJECT_MODEL_OUTPUT,
    "raw_barra_r2_lambda": RAW_BARRA_R2_LAMBDA,
    "adversarial_barra": ADVERSARIAL_BARRA,
    "adversarial_gamma_max": ADVERSARIAL_GAMMA_MAX,
    "barra_output_neutralization": BARRA_OUTPUT_NEUTRALIZATION,
    "barra_conditioning_mode": BARRA_CONDITIONING_MODE,
    "barra_dual_tower": BARRA_DUAL_TOWER,
    "barra_dual_tower_mode": BARRA_DUAL_TOWER_MODE,
    "dual_tower_alpha_loss_weight": DUAL_TOWER_ALPHA_LOSS_WEIGHT,
    "dual_tower_style_loss_weight": DUAL_TOWER_STYLE_LOSS_WEIGHT,
    "dual_tower_style_hidden_dim": DUAL_TOWER_STYLE_HIDDEN_DIM,
    "dual_tower_style_dropout": DUAL_TOWER_STYLE_DROPOUT,
    "barra_gate_hidden_dim": BARRA_GATE_HIDDEN_DIM,
    "barra_industry_embedding_dim": BARRA_INDUSTRY_EMBEDDING_DIM,
    "barra_gate_dropout": BARRA_GATE_DROPOUT,
    "barra_gate_scale": BARRA_GATE_SCALE,
    "loss": LOSS_NAME,
    "full_training": RUN_FULL_TRAINING,
    "launch_mode": TRAIN_LAUNCH_MODE,
    "screen_session": SCREEN_SESSION,
})


{'experiment_id': 'no6', 'title': 'Alpha/Style dual tower', 'barra_as_model_input': True, 'barra_in_loss': True}
{'training_route': 'ddp', 'run_name': 'dual_tower_ddp_ddp', 'backend': 'stage06_barra_training.py', 'residualize_target': False, 'project_model_output': False, 'raw_barra_r2_lambda': 0.3, 'adversarial_barra': False, 'adversarial_gamma_max': 0.1, 'barra_output_neutralization': False, 'barra_conditioning_mode': 'off', 'barra_dual_tower': True, 'barra_dual_tower_mode': 'real', 'dual_tower_alpha_loss_weight': 1.0, 'dual_tower_style_loss_weight': 1.0, 'dual_tower_style_hidden_dim': 64, 'dual_tower_style_dropout': 0.1, 'barra_gate_hidden_dim': 64, 'barra_industry_embedding_dim': 16, 'barra_gate_dropout': 0.1, 'barra_gate_scale': 0.5, 'loss': 'sigmoid_rank_ic', 'full_training': True, 'launch_mode': 'background', 'screen_session': 'vivit_dual_tower_ddp_ddp'}


## 步骤 2：准备风格数据并运行真实 batch smoke

No.6 会构建或复用三类对齐数据：十个连续风格/反转暴露与有效掩码、行业代码、完整 Barra 标签分解。完整残差标签继续监督 Alpha 塔；新增 fitted-target sidecar 只保存标准化后的 Barra 可解释收益并监督 Style 塔。构建时检查原始收益可由拟合项与残差精确重构，并检查拟合项位于完整风险列空间。

Smoke 使用 2018、seed 42 的一个完整日截面执行双塔前向、四项损失、反向传播和验证；不写正式 checkpoint 或样本外预测。


In [2]:
if STYLE_MODE:
    prepared = subprocess.run(
        [
            str(PYTHON), str(BACKEND), "prepare-data",
            "--overwrite", "0",
            "--include-industry-sidecar", str(int(R2_MODE or ADVERSARIAL_MODE or NO4_MODE or NO6_MODE)),
            "--include-full-barra-target", str(int(NO4_MODE or NO6_MODE)),
            "--include-dual-tower-target", str(int(NO6_MODE)),
        ],
        cwd=VIVIT_DIR, check=True, text=True, capture_output=True,
    )
    print(prepared.stdout)

if RUN_SMOKE_TEST:
    smoke_args = TRAINING_ARGS.copy()
    for option, value in [("--compile", "0"), ("--workers-per-task", "0")]:
        smoke_args[smoke_args.index(option) + 1] = value
    smoke_route = "ddp-smoke" if TRAINING_ROUTE == "ddp" else "smoke"
    smoke_device_args = ["--device-ids", DEVICE_IDS] if TRAINING_ROUTE == "ddp" else []
    smoke = subprocess.run(
        [
            str(PYTHON), str(BACKEND), smoke_route, *smoke_args,
            "--forecast-year", "2018", "--seed", "42", "--max-batches", "1",
            *smoke_device_args,
        ],
        cwd=VIVIT_DIR, check=True, text=True, capture_output=True,
    )
    print(smoke.stdout)
else:
    print("RUN_SMOKE_TEST=False，跳过 smoke。")


{
  "style_store": {
    "signature": "91aa60eaa0b3346666fe6296f8a9fc1c985b49ad927150c116bdf083e0e9970c",
    "payload": {
      "schema_version": 1,
      "source_dataset_signature": "7e549f6c4204f6228867617461dd8402bc1868e21b320239cdd8c262ad5f5058",
      "sample_index": {
        "path": "/root/autodl-tmp/ViT/data/feature_store/sample_index.parquet",
        "size": 36516427,
        "mtime_ns": 1785224380728501732
      },
      "source_targets": {
        "path": "/root/autodl-tmp/ViT/data/feature_store/targets.npy",
        "size": 31387944,
        "mtime_ns": 1785224377836304441
      },
      "cne5s": {
        "path": "/root/autodl-tmp/CNE5S.parquet",
        "size": 2176192699,
        "mtime_ns": 1784701309598925847,
        "rows": 17600174,
        "row_groups": 144,
        "columns": [
          "datadate",
          "barrid",
          "shares",
          "assetid",
          "price",
          "capt",
          "pricesource",
          "currency",
          "dlyreturn

## 步骤 3：持久化启动 8 年度 × 3 种子 A2/DDP 训练

A2 按 GPU 并发独立任务；DDP 则让每个年度 × seed 使用 DEVICE_IDS 中全部 GPU，24 个任务顺序执行。2025 只训练与验证，不生成样本外预测文件。`background` 模式会为每个 run 创建独立的 detached screen。

重复执行时，活动任务不会重复启动；已完成任务直接复用，非完整任务从 `last.pt` 续训。No.6 的 checkpoint 保存 ViViT、Alpha 条件门控和独立 Style 塔；模型结构、四项损失权重、输入模式、两个标签 sidecar 或集成尺度变化时，旧 checkpoint 不会被静默复用。


In [3]:
training_command = [
    str(PYTHON), str(BACKEND), TRAINING_ROUTE, *TRAINING_ARGS,
    "--device-ids", DEVICE_IDS,
    *(["--tasks-per-gpu", str(TASKS_PER_GPU)] if TRAINING_ROUTE == "a2" else []),
]
print("训练命令：", " ".join(training_command))
launch_result = None
if RUN_FULL_TRAINING:
    if TRAIN_LAUNCH_MODE == "background":
        launch_result = launch_training_job(
            training_command, RUN_NAME,
            screen_session_name=SCREEN_SESSION,
            allow_command_change=bool(OVERWRITE),
            restart_completed=bool(OVERWRITE),
            vivit_dir=VIVIT_DIR,
        )
        print("launch_action:", launch_result["launch_action"])
        print(format_training_job_status(launch_result))
        if launch_result["screen_alive"]:
            print(f"查看后台终端：screen -r {SCREEN_SESSION}")
    else:
        print("警告：foreground 模式仍依赖当前 Notebook kernel。")
        subprocess.run(training_command, cwd=VIVIT_DIR, check=True)
        launch_result = training_job_status(
            RUN_NAME, screen_session_name=SCREEN_SESSION, vivit_dir=VIVIT_DIR,
        )
        print(format_training_job_status(launch_result))
else:
    print("RUN_FULL_TRAINING=False，本次未启动正式训练；可在步骤4查看已有状态。")


训练命令： /root/miniconda3/bin/python /root/autodl-tmp/ViViT/stage06_barra_training.py ddp --run-name dual_tower_ddp_ddp --train-batch-mode trade_date --loss-name sigmoid_rank_ic --rank-temperature 0.1 --turnover-lambda 0.1 --huber-delta 1.0 --smooth-l1-beta 0.1 --hybrid-weight 0.5 --validation-metric daily_mean_spearman_ic --workers-per-task 8 --learning-rate 0.001 --max-epochs 100 --patience 10 --compile 1 --compile-mode reduce-overhead --fused-adam 1 --cuda-prefetch 1 --overwrite 0 --residualize-target 0 --project-model-output 0 --raw-barra-r2-lambda 0.3 --barra-conditioning-mode off --barra-output-neutralization 0 --barra-dual-tower 1 --barra-dual-tower-mode real --dual-tower-alpha-loss-weight 1.0 --dual-tower-style-loss-weight 1.0 --dual-tower-style-hidden-dim 64 --dual-tower-style-dropout 0.1 --barra-gate-hidden-dim 64 --barra-industry-embedding-dim 16 --barra-gate-dropout 0.1 --barra-gate-scale 0.5 --adversarial-barra 0 --adversarial-gamma-max 0.1 --adversarial-warmup-epochs 2 --adv

## 步骤 4：查看后台进度，完成后核验产物

本单元可在断线重连后反复执行。若 kernel 已重启，只需重新运行步骤 1 恢复参数，不必重复启动已有任务。

No.6 完成后会核验 24 个任务、2018–2024 七个预测文件、两个标签 sidecar 签名、双塔结构参数和逐日加法恒等式。正式主列必须等于 Alpha 集成列；Alpha、Style、Total 三类分数分别保存诊断，且 `Total = Alpha + Style` 的数值误差必须在容差内。


In [5]:
current_status = training_job_status(
    RUN_NAME, screen_session_name=SCREEN_SESSION, vivit_dir=VIVIT_DIR,
)
print(format_training_job_status(current_status))
log_tail = tail_training_job_log(
    RUN_NAME, lines=STATUS_LOG_TAIL_LINES, vivit_dir=VIVIT_DIR,
)
if log_tail:
    print("\n后台总控日志末尾：")
    print(log_tail)

if current_status["status"] in {"completed", "completed_existing"}:
    validation_args = TRAINING_ARGS if STYLE_MODE else ["--run-name", RUN_NAME]
    validated = subprocess.run(
        [str(PYTHON), str(BACKEND), "validate", *validation_args],
        cwd=VIVIT_DIR, check=True, text=True, capture_output=True,
    )
    print(validated.stdout)
    reported = subprocess.run(
        [str(PYTHON), str(BACKEND), "report", *validation_args],
        cwd=VIVIT_DIR, check=True, text=True, capture_output=True,
    )
    print(reported.stdout)
    if ADVERSARIAL_MODE:
        probe_path = VIVIT_DIR / "data" / "quality" / "runs" / RUN_NAME / "adversarial_probe_diagnostics.parquet"
        print("冻结 probe 汇总：", probe_path)
else:
    print("后台训练尚未完整结束，本次跳过产物核验；稍后重新运行本单元。")
    if current_status["status"] in {"failed", "interrupted", "resumable", "incomplete_outputs"}:
        print("任务可恢复：保持同一 RUN_NAME 和 OVERWRITE=0，重新运行步骤3。")


run_name: dual_tower_ddp_ddp
status: completed
screen_session: vivit_dual_tower_ddp_ddp
screen_alive: False
tasks: completed=24/24, running=0, resumable=0
prediction_files: 7
log: /root/autodl-tmp/ViViT/data/quality/runs/dual_tower_ddp_ddp/background_job/orchestrator.log
last_heartbeat: 2026-08-19T01:51:05.169680+00:00
exit_code: 0
 year    rows status
 2018  796559     通过
 2019  861870     通过
 2020  916587     通过
 2021 1027544     通过
 2022 1120684     通过
 2023 1195010     通过
 2024 1200893     通过
 2025       0     通过

/root/autodl-tmp/ViViT/data/quality/runs/dual_tower_ddp_ddp/EXPERIMENT_REPORT.md



## 步骤 5：接入 Notebook03/04 评价

No.6 的默认正式信号是未经投影的 Alpha 塔集成分数，因此原有主列可以直接回测。若要拆解双塔，也可以在评价 Notebook 的预测列参数中选择 Alpha、Style 或 Total；三类评价产物会写入彼此隔离的目录，不会互相覆盖。


In [5]:
FOLDER_NAME = RUN_NAME
BASELINE_RUN_NAME = "sigmoid_daybatch_rankic"
BACKTEST_MODE = "mode2_t_no_filter"
print({
    "FOLDER_NAME": FOLDER_NAME,
    "BASELINE_RUN_NAME": BASELINE_RUN_NAME,
    "BACKTEST_MODE": BACKTEST_MODE,
})


{'FOLDER_NAME': 'dual_tower_ddp_ddp', 'BASELINE_RUN_NAME': 'sigmoid_daybatch_rankic', 'BACKTEST_MODE': 'mode2_t_no_filter'}
